## Task 01. Nested Transaction Analyzer

In [1]:
def analyze_transactions(transactions):
    ids = [t["id"] for t in transactions]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate transaction IDs found")
    report = {}
    for t in transactions:
        c = report.setdefault(t["customer"], {"total": 0, "cats": {}})
        if t["status"] == "success":
            c["total"] += t["amount"]
            c["cats"][t["category"]] = c["cats"].get(t["category"], 0) + t["amount"]
    rows = [{"customer": name, "total": d["total"],
             "top_category": min(d["cats"], key=lambda k: (-d["cats"][k], k)) if d["cats"] else None}
            for name, d in report.items()]
    rows.sort(key=lambda r: (-r["total"], r["customer"]))
    for i, r in enumerate(rows, 1):
        r["rank"] = i
    return rows


data = [
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
    {"id": 4, "customer": "Zain", "category": "Food", "amount": 300, "status": "failed"},
]
for row in analyze_transactions(data):
    print(row)

try:
    analyze_transactions(data + [{"id": 1, "customer": "Ali", "category": "Food", "amount": 50, "status": "success"}])
except ValueError as e:
    print("Error:", e)

{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}
{'customer': 'Zain', 'total': 0, 'top_category': None, 'rank': 3}
Error: Duplicate transaction IDs found


## Task 02. Longest Valid Window

In [2]:
def longest_unique_window(text):
    last_seen, start, best_start, best_len = {}, 0, 0, 0
    for i, ch in enumerate(text):
        if last_seen.get(ch, -1) >= start:
            start = last_seen[ch] + 1
        last_seen[ch] = i
        if i - start + 1 > best_len:
            best_start, best_len = start, i - start + 1
    return text[best_start:best_start + best_len], best_start, best_len


for s in ["abcabcbb", "", "bbbbb", "pwwkew", "a b!a b"]:
    print(repr(s), "->", longest_unique_window(s))

'abcabcbb' -> ('abc', 0, 3)
'' -> ('', 0, 0)
'bbbbb' -> ('b', 0, 1)
'pwwkew' -> ('wke', 2, 3)
'a b!a b' -> ('a b!', 0, 4)


## Task 03. Recursive JSON Path Explorer

In [3]:
def find_paths(data, target, path=""):
    found = []
    if isinstance(data, dict):
        for key, value in data.items():
            found += find_paths(value, target, f"{path}.{key}" if path else str(key))
    elif isinstance(data, list):
        for i, value in enumerate(data):
            found += find_paths(value, target, f"{path}[{i}]")
    elif data == target and isinstance(data, bool) == isinstance(target, bool):
        found.append(path)
    return found


sample = {"a": [{"x": 5}, 5], "b": {"y": 5}}
print(find_paths(sample, 5))
print(find_paths({"flag": True, "n": 1, "l": [1, True]}, 1))
print(find_paths({"flag": True, "l": [1, True]}, True))

['a[0].x', 'a[1]', 'b.y']
['n', 'l[0]']
['flag', 'l[1]']


## Task 04. Dependency Execution Planner

In [4]:
import heapq


def execution_order(tasks, dependencies):
    graph = {t: [] for t in tasks}
    indegree = {t: 0 for t in graph}
    for a, b in dependencies:
        if a not in graph or b not in graph:
            raise ValueError(f"Unknown task in dependency ({a}, {b})")
        graph[a].append(b)
        indegree[b] += 1
    ready = [t for t in graph if indegree[t] == 0]
    heapq.heapify(ready)
    order = []
    while ready:
        task = heapq.heappop(ready)
        order.append(task)
        for nxt in graph[task]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                heapq.heappush(ready, nxt)
    if len(order) != len(graph):
        raise ValueError("Cycle detected: tasks cannot be ordered")
    return order


print(execution_order(["build", "test", "deploy"], [("build", "test"), ("test", "deploy")]))
print(execution_order(["c", "b", "a"], []))

for tasks, deps in [(["a", "b"], [("a", "b"), ("b", "a")]), (["a"], [("a", "z")])]:
    try:
        execution_order(tasks, deps)
    except ValueError as e:
        print("Error:", e)

['build', 'test', 'deploy']
['a', 'b', 'c']
Error: Cycle detected: tasks cannot be ordered
Error: Unknown task in dependency (a, z)


## Task 05. Expression Validator and Evaluator

In [5]:
import re
import operator

FUNCS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
PRIORITY = {"+": 1, "-": 1, "*": 2, "/": 2, "u": 3}


def calculate(expression):
    nums, ops = [], []

    def apply():
        op = ops.pop()
        if op == "u":
            nums.append(-nums.pop())
            return
        b, a = nums.pop(), nums.pop()
        if op == "/" and b == 0:
            raise ZeroDivisionError("Division by zero")
        nums.append(FUNCS[op](a, b))

    expect_value = True
    for t in re.findall(r"\d+|\S", expression):
        if t.isdigit():
            if not expect_value:
                raise ValueError("Malformed expression")
            nums.append(int(t))
            expect_value = False
        elif t == "(":
            if not expect_value:
                raise ValueError("Malformed expression")
            ops.append(t)
        elif t == ")":
            if expect_value:
                raise ValueError("Malformed expression")
            while ops and ops[-1] != "(":
                apply()
            if not ops:
                raise ValueError("Mismatched parentheses")
            ops.pop()
        elif t in FUNCS:
            if expect_value:
                if t != "-":
                    raise ValueError("Malformed expression")
                ops.append("u")
                continue
            while ops and ops[-1] != "(" and PRIORITY[ops[-1]] >= PRIORITY[t]:
                apply()
            ops.append(t)
            expect_value = True
        else:
            raise ValueError(f"Invalid character: {t}")
    if expect_value:
        raise ValueError("Malformed expression")
    while ops:
        if ops[-1] == "(":
            raise ValueError("Mismatched parentheses")
        apply()
    return float(nums[0])


for e in ["-(3 + 2) * 4 + 18 / 3", "2 + 3 * 4", "(2 + 3) * 4", "2 * -3", "10 / 0", "2 +", "(2 + 3", "2 $ 3"]:
    try:
        print(e, "=", calculate(e))
    except (ValueError, ZeroDivisionError) as err:
        print(e, "-> Error:", err)

-(3 + 2) * 4 + 18 / 3 = -14.0
2 + 3 * 4 = 14.0
(2 + 3) * 4 = 20.0
2 * -3 = -6.0
10 / 0 -> Error: Division by zero
2 + -> Error: Malformed expression
(2 + 3 -> Error: Mismatched parentheses
2 $ 3 -> Error: Invalid character: $


## Task 06. Banking Ledger with OOP

In [6]:
import itertools
from dataclasses import dataclass


@dataclass(frozen=True)
class Transaction:
    id: int
    action: str
    amount: float
    balance: float


class Account:
    _ids = itertools.count(1)

    def __init__(self, name, balance=0):
        self.name = name
        self._balance = 0
        self._history = []
        if balance:
            self.deposit(balance, "opening deposit")

    @property
    def balance(self):
        return self._balance

    def _record(self, action, amount):
        self._history.append(Transaction(next(Account._ids), action, amount, self._balance))

    def deposit(self, amount, action="deposit"):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self._balance += amount
        self._record(action, amount)

    def withdraw(self, amount, action="withdraw"):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        self._record(action, amount)

    def statement(self):
        return list(self._history)


class Bank:
    def __init__(self):
        self.accounts = {}

    def open_account(self, name, balance=0):
        self.accounts[name] = Account(name, balance)

    def _get(self, name):
        if name not in self.accounts:
            raise ValueError(f"Account {name} does not exist")
        return self.accounts[name]

    def transfer(self, src, dst, amount):
        a, b = self._get(src), self._get(dst)
        if src == dst:
            raise ValueError("Cannot transfer to the same account")
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > a.balance:
            raise ValueError("Insufficient funds")
        a.withdraw(amount, f"transfer to {dst}")
        b.deposit(amount, f"transfer from {src}")


bank = Bank()
bank.open_account("A", 1000)
bank.open_account("B", 200)
bank.transfer("A", "B", 350)
print("A =", bank.accounts["A"].balance, "| B =", bank.accounts["B"].balance)

for src, dst, amt in [("A", "B", 5000), ("A", "C", 10), ("A", "B", -5)]:
    try:
        bank.transfer(src, dst, amt)
    except ValueError as e:
        print("Error:", e)
print("A =", bank.accounts["A"].balance, "| B =", bank.accounts["B"].balance)

for t in bank.accounts["B"].statement():
    print(t)

A = 650 | B = 550
Error: Insufficient funds
Error: Account C does not exist
Error: Amount must be positive
A = 650 | B = 550
Transaction(id=2, action='opening deposit', amount=200, balance=200)
Transaction(id=4, action='transfer from A', amount=350, balance=550)


## Task 07. Employee Payroll Engine

In [7]:
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        pass


class SalariedEmployee(Employee):
    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = salary

    def calculate_pay(self):
        return self.salary


class HourlyEmployee(Employee):
    def __init__(self, name, rate, hours):
        super().__init__(name)
        self.rate, self.hours = rate, hours

    def calculate_pay(self):
        overtime = max(0, self.hours - 40)
        return self.rate * (self.hours - overtime) + self.rate * 1.5 * overtime


class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name)
        self.base, self.sales, self.rate = base, sales, rate

    def calculate_pay(self):
        return self.base + self.sales * self.rate


def payroll_summary(employees):
    total = 0
    for e in employees:
        pay = e.calculate_pay()
        total += pay
        print(f"{e.name:<8}{type(e).__name__:<20}{pay:>10.2f}")
    print(f"{'TOTAL':<28}{total:>10.2f}")


payroll_summary([
    SalariedEmployee("Ayesha", 3000),
    HourlyEmployee("Bilal", 20, 45),
    CommissionEmployee("Chand", 500, 4000, 0.10),
])

try:
    Employee("Test")
except TypeError as e:
    print("Error:", e)

Ayesha  SalariedEmployee       3000.00
Bilal   HourlyEmployee          950.00
Chand   CommissionEmployee      900.00
TOTAL                          4850.00
Error: Can't instantiate abstract class Employee without an implementation for abstract method 'calculate_pay'


## Task 08. Cache Decorator with Expiration

In [8]:
import time
import functools


def ttl_cache(seconds):
    def decorator(func):
        cache = {}
        stats = {"hits": 0, "misses": 0}

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            try:
                hash(key)
            except TypeError:
                stats["misses"] += 1
                return func(*args, **kwargs)
            now = time.monotonic()
            if key in cache and now - cache[key][1] < seconds:
                stats["hits"] += 1
                return cache[key][0]
            stats["misses"] += 1
            cache[key] = (func(*args, **kwargs), now)
            return cache[key][0]

        wrapper.stats = stats
        return wrapper
    return decorator


@ttl_cache(1)
def add(x, y):
    """Add two numbers"""
    return x + y


add(x=2, y=3)
add(y=3, x=2)
print(add.stats)

time.sleep(1.1)
add(x=2, y=3)
print(add.stats)

print(add([1], [2]), add.stats)
print(add.__name__, "|", add.__doc__)

{'hits': 1, 'misses': 1}
{'hits': 1, 'misses': 2}
[1, 2] {'hits': 1, 'misses': 3}
add | Add two numbers


## Task 09. Streaming Log Inspector

In [9]:
from datetime import datetime
from collections import Counter

with open("app.log", "w") as f:
    f.write("2026-11-07 10:30:00 | ERROR | Connection lost\n"
            "2026-11-07 10:31:00 | INFO | Reconnected\n"
            "this line is broken\n"
            "2026-11-07 11:00:00 | ERROR | Disk full | extra info\n"
            "2026-11-08 09:15:00 | ERROR | Timeout\n"
            "2026-13-45 09:15:00 | ERROR | Bad date\n")


def read_errors(path, stats=None):
    stats = stats if stats is not None else {}
    stats.setdefault("malformed", 0)
    with open(path) as f:
        for number, line in enumerate(f, 1):
            parts = [p.strip() for p in line.split("|", 2)]
            try:
                if len(parts) != 3:
                    raise ValueError
                datetime.strptime(parts[0], "%Y-%m-%d %H:%M:%S")
            except ValueError:
                stats["malformed"] += 1
                continue
            if parts[1] == "ERROR":
                yield number, parts[0], parts[2]


def summarize_errors(path):
    stats = {}
    counts = Counter(ts[:10] for _, ts, _ in read_errors(path, stats))
    return dict(counts), stats["malformed"]


for record in read_errors("app.log"):
    print(record)
print(summarize_errors("app.log"))

(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 11:00:00', 'Disk full | extra info')
(5, '2026-11-08 09:15:00', 'Timeout')
({'2026-11-07': 2, '2026-11-08': 1}, 2)


## Task 10. CSV Inventory Reconciliation

In [10]:
import csv

with open("inventory.csv", "w", newline="") as f:
    f.write("sku,name,quantity\nP01,Pen,10\nP02,Notebook,5\n")
with open("movements.csv", "w", newline="") as f:
    f.write("sku,type,units\nP01,OUT,4\nP01,OUT,9\nP01,IN,2\nX99,IN,5\nP02,IN,-3\nP02,MOVE,2\nP02,OUT,5\n")


def reconcile(inventory_file, movements_file):
    with open(inventory_file, newline="") as f:
        items = list(csv.DictReader(f))
    stock = {i["sku"]: int(i["quantity"]) for i in items}
    rejected = []
    with open(movements_file, newline="") as f:
        for m in csv.DictReader(f):
            try:
                units, kind = int(m["units"]), m["type"].strip().upper()
                if m["sku"] not in stock:
                    raise ValueError("unknown SKU")
                if units <= 0:
                    raise ValueError("units must be positive")
                if kind not in ("IN", "OUT"):
                    raise ValueError("invalid movement type")
                if kind == "OUT" and units > stock[m["sku"]]:
                    raise ValueError("stock would go negative")
                stock[m["sku"]] += units if kind == "IN" else -units
            except ValueError as e:
                rejected.append({**m, "reason": e})
    for i in items:
        i["quantity"] = stock[i["sku"]]
    for name, fields, rows in [("updated_inventory.csv", ["sku", "name", "quantity"], items),
                               ("rejected_movements.csv", ["sku", "type", "units", "reason"], rejected)]:
        with open(name, "w", newline="") as f:
            writer = csv.DictWriter(f, fields)
            writer.writeheader()
            writer.writerows(rows)


reconcile("inventory.csv", "movements.csv")
print(open("updated_inventory.csv").read())
print(open("rejected_movements.csv").read())

sku,name,quantity
P01,Pen,8
P02,Notebook,0

sku,type,units,reason
P01,OUT,9,stock would go negative
X99,IN,5,unknown SKU
P02,IN,-3,units must be positive
P02,MOVE,2,invalid movement type



## Task 11. Multi-File Student Gradebook

In [11]:
import csv

with open("students.csv", "w", newline="") as f:
    f.write("student_id,name\n101,Ali\n102,Sara\n103,Zain\n")
with open("scores.csv", "w", newline="") as f:
    f.write("student_id,subject,marks\n101,Math,90\n101,Physics,80\n101,English,85\n"
            "102,Math,95\n102,Physics,88\n103,Math,45\n103,Math,60\n999,Math,70\n102,English,150\n")


def grade(avg):
    return next(g for cut, g in [(85, "A"), (70, "B"), (55, "C"), (40, "D"), (0, "F")] if avg >= cut)


def build_report(students_file, scores_file, out_file="report.csv"):
    with open(students_file, newline="") as f:
        names = {r["student_id"]: r["name"] for r in csv.DictReader(f)}
    marks, seen, rejected = {}, set(), []
    with open(scores_file, newline="") as f:
        for r in csv.DictReader(f):
            try:
                m, key = float(r["marks"]), (r["student_id"], r["subject"])
                if r["student_id"] not in names:
                    raise ValueError("unknown student ID")
                if not 0 <= m <= 100:
                    raise ValueError("marks must be 0-100")
                if key in seen:
                    raise ValueError("duplicate student-subject entry")
            except ValueError as e:
                rejected.append((r["student_id"], r["subject"], str(e)))
                continue
            seen.add(key)
            marks.setdefault(r["student_id"], []).append(m)
    avgs = {sid: sum(v) / len(v) for sid, v in marks.items()}
    ranked = sorted(avgs, key=lambda sid: (-avgs[sid], sid))
    with open(out_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["student_id", "name", "average", "grade", "rank"])
        for rank, sid in enumerate(ranked, 1):
            writer.writerow([sid, names[sid], round(avgs[sid], 2), grade(avgs[sid]), rank])
    return rejected


rejected = build_report("students.csv", "scores.csv")
print(open("report.csv").read())
print("Rejected:", rejected)

student_id,name,average,grade,rank
102,Sara,91.5,A,1
101,Ali,85.0,A,2
103,Zain,45.0,D,3

Rejected: [('103', 'Math', 'duplicate student-subject entry'), ('999', 'Math', 'unknown student ID'), ('102', 'English', 'marks must be 0-100')]


## Task 12. Custom Iterable Range Toolkit

In [12]:
class ReversibleRange:
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("step cannot be zero")
        self.start, self.stop, self.step = start, stop, step

    def __len__(self):
        adjust = -1 if self.step > 0 else 1
        return max(0, (self.stop - self.start + self.step + adjust) // self.step)

    def __iter__(self):
        x = self.start
        while x < self.stop if self.step > 0 else x > self.stop:
            yield x
            x += self.step

    def __reversed__(self):
        n = len(self)
        x = self.start + (n - 1) * self.step
        while n:
            yield x
            x -= self.step
            n -= 1

    def __contains__(self, value):
        if (value - self.start) % self.step:
            return False
        return self.start <= value < self.stop if self.step > 0 else self.stop < value <= self.start


r = ReversibleRange(2, 11, 3)
print(list(r), list(reversed(r)), len(r), 5 in r, 6 in r)

r = ReversibleRange(10, 0, -3)
print(list(r), list(reversed(r)), len(r), 4 in r, 5 in r)

print(list(ReversibleRange(5, 0)), len(ReversibleRange(5, 0)))

[2, 5, 8] [8, 5, 2] 3 True False
[10, 7, 4, 1] [1, 4, 7, 10] 4 True False
[] 0


## Task 13. Booking Conflict Scheduler

In [13]:
from datetime import datetime


class Scheduler:
    def __init__(self):
        self.bookings = []

    def book(self, room, start, end, booking_id):
        if end <= start:
            raise ValueError("End time must be after start time")
        if any(b["id"] == booking_id for b in self.bookings):
            raise ValueError("Duplicate booking ID")
        conflicts = [b["id"] for b in self.bookings
                     if b["room"] == room and start < b["end"] and b["start"] < end]
        if conflicts:
            return False, conflicts
        self.bookings.append({"id": booking_id, "room": room, "start": start, "end": end})
        self.bookings.sort(key=lambda b: (b["room"], b["start"]))
        return True, []


def t(hhmm):
    return datetime.strptime(f"2026-11-07 {hhmm}", "%Y-%m-%d %H:%M")


s = Scheduler()
print(s.book("A", t("09:00"), t("10:00"), "B1"))
print(s.book("A", t("10:00"), t("11:00"), "B2"))
print(s.book("A", t("09:30"), t("10:30"), "B3"))
print(s.book("B", t("09:30"), t("10:30"), "B4"))

for args in [("A", t("12:00"), t("11:00"), "B5"), ("A", t("12:00"), t("13:00"), "B1")]:
    try:
        s.book(*args)
    except ValueError as e:
        print("Error:", e)

(True, [])
(True, [])
(False, ['B1', 'B2'])
(True, [])
Error: End time must be after start time
Error: Duplicate booking ID


## Task 14. Command-Line Contact Manager

In [14]:
import json
import os
import re

FILE = "contacts.json"


def load_contacts():
    if not os.path.exists(FILE):
        return []
    try:
        with open(FILE) as f:
            return json.load(f)
    except json.JSONDecodeError:
        os.replace(FILE, FILE + ".corrupt")
        print("Corrupt file saved as", FILE + ".corrupt")
        return []


def save_contacts(contacts):
    with open(FILE + ".tmp", "w") as f:
        json.dump(contacts, f, indent=2)
    os.replace(FILE + ".tmp", FILE)


def validate(name, phone, email):
    if not name.strip():
        raise ValueError("Name is required")
    if not re.fullmatch(r"\+?\d{7,15}", re.sub(r"[\s-]", "", phone)):
        raise ValueError("Invalid phone number")
    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.\w+", email):
        raise ValueError("Invalid email")


def add_contact(name, phone, email):
    validate(name, phone, email)
    contacts = load_contacts()
    new_id = max((c["id"] for c in contacts), default=0) + 1
    contacts.append({"id": new_id, "name": name.strip(), "phone": phone, "email": email})
    save_contacts(contacts)
    return new_id


def search_contacts(query):
    return [c for c in load_contacts() if query.lower() in c["name"].lower()]


def update_contact(contact_id, **changes):
    contacts = load_contacts()
    contact = next((c for c in contacts if c["id"] == contact_id), None)
    if contact is None:
        raise ValueError("Contact not found")
    updated = {**contact, **changes}
    validate(updated["name"], updated["phone"], updated["email"])
    contact.update(updated)
    save_contacts(contacts)


def delete_contact(contact_id):
    contacts = load_contacts()
    remaining = [c for c in contacts if c["id"] != contact_id]
    if len(remaining) == len(contacts):
        raise ValueError("Contact not found")
    save_contacts(remaining)


def main():
    while True:
        choice = input("1.Add 2.Search 3.Update 4.Delete 5.List 6.Exit: ").strip()
        try:
            if choice == "1":
                print("Added with ID", add_contact(input("Name: "), input("Phone: "), input("Email: ")))
            elif choice == "2":
                print(*(search_contacts(input("Search: ")) or ["No match"]), sep="\n")
            elif choice == "3":
                update_contact(int(input("ID: ")), name=input("Name: "), phone=input("Phone: "), email=input("Email: "))
            elif choice == "4":
                delete_contact(int(input("ID: ")))
            elif choice == "5":
                print(*(load_contacts() or ["No contacts"]), sep="\n")
            elif choice == "6":
                break
        except ValueError as e:
            print("Error:", e)


if os.path.exists(FILE):
    os.remove(FILE)

print("Added ID:", add_contact("Ayesha", "0300-1234567", "ayesha@mail.com"))
print("Search 'aye':", search_contacts("aye"))
update_contact(1, phone="03111234567")
print("After update:", search_contacts("AYE"))

try:
    add_contact("Bad", "abc", "no-email")
except ValueError as e:
    print("Error:", e)

delete_contact(1)
print("After delete:", load_contacts())

Added ID: 1
Search 'aye': [{'id': 1, 'name': 'Ayesha', 'phone': '0300-1234567', 'email': 'ayesha@mail.com'}]
After update: [{'id': 1, 'name': 'Ayesha', 'phone': '03111234567', 'email': 'ayesha@mail.com'}]
Error: Invalid phone number
After delete: []


## Task 15. Integrated Order Fulfillment System

In [15]:
import json
import os
import tempfile
import unittest
import uuid
from dataclasses import dataclass


@dataclass(frozen=True)
class Product:
    id: str
    name: str
    price: float


class Cart:
    def __init__(self, discount=0):
        if not 0 <= discount <= 100:
            raise ValueError("Discount must be between 0 and 100")
        self.items, self.discount = {}, discount

    def add(self, product_id, qty):
        if not isinstance(qty, int) or qty <= 0:
            raise ValueError("Quantity must be a positive integer")
        self.items[product_id] = self.items.get(product_id, 0) + qty


class Order:
    def __init__(self, cart, products):
        self.id = uuid.uuid4().hex[:8]
        self.items = dict(cart.items)
        self.discount = cart.discount
        subtotal = sum(products[pid].price * q for pid, q in self.items.items())
        self.total = round(subtotal * (1 - cart.discount / 100), 2)

    def to_dict(self):
        return {"id": self.id, "items": self.items, "discount": self.discount, "total": self.total}


class Inventory:
    def __init__(self, catalog_path):
        with open(catalog_path) as f:
            data = json.load(f)
        self.products = {p["id"]: Product(p["id"], p["name"], p["price"]) for p in data}
        self.stock = {p["id"]: p["stock"] for p in data}

    def place_order(self, cart, orders_path):
        for pid, qty in cart.items.items():
            if pid not in self.products:
                raise ValueError(f"Unknown product {pid}")
            if qty > self.stock[pid]:
                raise ValueError(f"Out of stock: {pid}")
        order = Order(cart, self.products)
        backup = dict(self.stock)
        for pid, qty in cart.items.items():
            self.stock[pid] -= qty
        try:
            save_order(order, orders_path)
        except Exception:
            self.stock = backup
            raise
        return order


def save_order(order, path):
    orders = json.load(open(path)) if os.path.exists(path) else []
    orders.append(order.to_dict())
    with open(path + ".tmp", "w") as f:
        json.dump(orders, f, indent=2)
    os.replace(path + ".tmp", path)


class TestOrders(unittest.TestCase):
    def setUp(self):
        self.dir = tempfile.mkdtemp()
        self.catalog = os.path.join(self.dir, "catalog.json")
        self.orders = os.path.join(self.dir, "orders.json")
        with open(self.catalog, "w") as f:
            json.dump([{"id": "P1", "name": "Pen", "price": 100, "stock": 5}], f)
        self.inv = Inventory(self.catalog)

    def test_total_and_stock_update(self):
        cart = Cart(discount=10)
        cart.add("P1", 2)
        order = self.inv.place_order(cart, self.orders)
        self.assertEqual(order.total, 180)
        self.assertEqual(self.inv.stock["P1"], 3)
        self.assertEqual(len(json.load(open(self.orders))), 1)

    def test_out_of_stock_rejected(self):
        cart = Cart()
        cart.add("P1", 10)
        with self.assertRaises(ValueError):
            self.inv.place_order(cart, self.orders)
        self.assertEqual(self.inv.stock["P1"], 5)

    def test_invalid_quantity(self):
        for bad in (0, -1, 2.5):
            with self.assertRaises(ValueError):
                Cart().add("P1", bad)

    def test_rollback_when_saving_fails(self):
        cart = Cart()
        cart.add("P1", 2)
        with self.assertRaises(OSError):
            self.inv.place_order(cart, os.path.join(self.dir, "missing", "orders.json"))
        self.assertEqual(self.inv.stock["P1"], 5)

    def test_unique_order_ids(self):
        ids = set()
        for _ in range(3):
            cart = Cart()
            cart.add("P1", 1)
            ids.add(self.inv.place_order(cart, self.orders).id)
        self.assertEqual(len(ids), 3)


unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(TestOrders))

test_invalid_quantity (__main__.TestOrders.test_invalid_quantity) ... ok
test_out_of_stock_rejected (__main__.TestOrders.test_out_of_stock_rejected) ... ok
test_rollback_when_saving_fails (__main__.TestOrders.test_rollback_when_saving_fails) ... ok
test_total_and_stock_update (__main__.TestOrders.test_total_and_stock_update) ... ok
test_unique_order_ids (__main__.TestOrders.test_unique_order_ids) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.094s

OK


<unittest.runner.TextTestResult run=5 errors=0 failures=0>